In [ ]:

import numpy as np
import torch
import gc
from tensorflow.keras import backend as K

import datetime

start_time = datetime.datetime.now()
today = datetime.date.today().strftime('%Y-%m-%d')
img_size = 30
Code_run_ID = today + 'no_onehot_encoding_run4'


2025-03-07 16:33:50.052371: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-07 16:33:50.108405: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-07 16:33:50.108455: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-07 16:33:50.110057: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-07 16:33:50.123113: I tensorflow/core/platform/cpu_feature_guar

In [ ]:
save_file_path = '/media/2tbdisk3/data/Haidee/Results/'

In [ ]:
if img_size == 30 or img_size == 20:
    training_data_path = '/home/ht21074/Auto_box_apples/Auto_box_apples/Source_folder/Training_data_May2025/30px/all_years/'
elif img_size == 50 or img_size ==40:
    training_data_path = '/home/ht21074/Auto_box_apples/Auto_box_apples/Source_folder/Training_data_May2025/50px/all_years/'

X_train_brix          = np.load(f'{training_data_path}X_train_all_years_Brix_shuffled.npy')
Y_train_brix          = np.load(f'{training_data_path}Y_train_all_years_Brix_shuffled.npy')
X_validate_brix       = np.load(f'{training_data_path}X_validate_all_years_Brix_shuffled.npy')
Y_validate_brix       = np.load(f'{training_data_path}Y_validate_all_years_Brix_shuffled.npy')
Brix_encoder_shuffled  = np.load(f'{training_data_path}X_train_all_years_Brix_encoder_shuffled.npy')
validate_encoder_brix        = np.load(f'{training_data_path}X_validate_all_years_Brix_encoder_shuffled.npy')

X_train_Firmness          = np.load(f'{training_data_path}X_train_all_years_Firmness_shuffled.npy')
Y_train_Firmness          = np.load(f'{training_data_path}Y_train_all_years_Firmness_shuffled.npy')
X_validate_Firmness       = np.load(f'{training_data_path}X_validate_all_years_Firmness_shuffled.npy')
Y_validate_Firmness       = np.load(f'{training_data_path}Y_validate_all_years_Firmness_shuffled.npy')
Frmness_encoder_shuffled  = np.load(f'{training_data_path}X_train_all_years_Firmness_encoder_shuffled.npy')
validate_encoder_firmness        = np.load(f'{training_data_path}X_validate_all_years_Firmness_encoder_shuffled.npy')



In [5]:
print(min(Y_train_Firmness))
print(max(Y_train_Firmness))

5.275
119.04


In [ ]:
print(f"Data type of X_train_brix: {X_train_brix.dtype}")
print(f"Data type of y_train_brix: {Y_train_brix.dtype}")
print(f"Data type of X_train_firmness: {X_train_Firmness.dtype}")
print(f"Data type of y_train_firmness: {Y_train_Firmness.dtype}")

Data type of X_train_brix: <U93
Data type of y_train_brix: <U78
Data type of X_train_brix: <U93
Data type of y_train_brix: float64


In [31]:
print(Y_train_brix)
print(Y_train_Firmness)

['12.08' '14.75' '12.7' ... '10.4' '10.97' '11.4']
[ 7.445   7.1    94.708  ... 87.4495  8.36   75.917 ]


In [6]:
print(X_train_brix[:3])

["subsetted_aggregated_hyperspectral_images/NZ2023/aggregated_Cox's Orange Pippin_1071_2758.npy"
 'subsetted_aggregated_hyperspectral_images/NZ2024/aggregated_Fuji_1325_7897.npy'
 'subsetted_aggregated_hyperspectral_images/UK2024/aggregated_Braeburn_1449_18312.npy']


In [ ]:
spectral_path = '/media/2tbdisk3/data/Haidee/May_2025_spectral_img/Spectral/'

X_train_brix = [spectral_path + file for file in X_train_brix]

print(X_train_brix[:3])

["/media/2tbdisk2/data/Haidee_apple_data/Haidee/Data_outputs_NZ2023/Spectral/subsetted_aggregated_hyperspectral_images/NZ2023/aggregated_Cox's Orange Pippin_1071_2758.npy", '/media/2tbdisk2/data/Haidee_apple_data/Haidee/Data_outputs_NZ2023/Spectral/subsetted_aggregated_hyperspectral_images/NZ2024/aggregated_Fuji_1325_7897.npy', '/media/2tbdisk2/data/Haidee_apple_data/Haidee/Data_outputs_NZ2023/Spectral/subsetted_aggregated_hyperspectral_images/UK2024/aggregated_Braeburn_1449_18312.npy']


In [4]:
import os

os.makedirs(Code_run_ID, exist_ok=True)

print(f"Directory '{Code_run_ID}' created successfully!")

Directory '2025-02-21run1' created successfully!


In [5]:

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    BatchNormalization,
    Dropout,
    GlobalAveragePooling2D,
)
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
from tensorflow.keras.callbacks import CSVLogger

print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))


Num GPUs Available:  2


In [6]:
from keras.backend import clear_session
import tensorflow

# Reset Keras Session # modified from https://forums.fast.ai/t/how-could-i-release-gpu-memory-of-keras/2023/17 Jaycangel
def reset_keras():
    # Clear keras session
    clear_session()

    # clear global variables
    try:
        del classifier  # Update the variable name as needed
    except NameError:
        pass

    # force garbage collect
    gc.collect()

    # reset the GPU memory
    tf.config.experimental.set_memory_growth(tf.config.list_physical_devices('GPU')[0], True)
    print("Keras session reset and GPU memory cleared.")


In [ ]:
# Clear the log file at the beginning if file exists

with open(f"{Code_run_ID}/{today}_missing_files.log", "w") as log_file:
    log_file.write("")  # Write an empty string to clear the file


In [ ]:
# Data generators help with managing memory when using large amounts of data

def data_generator_w_cultivar(file_list, targets, cultivars, batch_size, img_size):
    num_samples = len(file_list)
    missing_files = [] # List of missing files

    while True:  # Infinite loop for generator
        for offset in range(0, num_samples, batch_size):
            # Load the batch of data from file paths
            batch_files = file_list[offset: offset + batch_size]
            batch_data = []
            batch_targets = []
            batch_cultivars = []

            # File loading and handling - ensures model runs if file not found
            for i, file in enumerate(batch_files):
                try:
                    data = np.load(file)
                    batch_data.append(data)
                    batch_targets.append(targets[offset + i])
                    batch_cultivars.append(cultivars[offset + i])
                except FileNotFoundError:
                    missing_files.append(file)
                    print(f"File not found: {file}. Skipping...")
                    continue
            

            # Convert lists to numpy arrays
            batch_data = np.array(batch_data)  # Shape: (batch_size, 20, 20, 204)
            batch_targets = np.array(batch_targets)  # Shape: (batch_size,)
            batch_cultivars = np.array(batch_cultivars)  # Shape: (batch_size, 6)
            
            if len(batch_data) == 0:
                continue # Skip if no data loaded
            
                       
            # Expand cultivar information to match the input data's spatial dimensions
            expanded_cultivars = np.repeat(batch_cultivars[:, np.newaxis, np.newaxis, :], img_size, axis=1) # Adds singleton dimensions to match the input data's spatial dimensions (batchsize, 1, 1, 1, 6)
            expanded_cultivars = np.repeat(expanded_cultivars, img_size, axis=2)

            # Concatenate cultivar information with the original data along the last axis
            combined_data = np.concatenate([batch_data, expanded_cultivars], axis=-1)  # Shape: (batch_size, 20, 20, 204, 6)


            # Yield the combined data and targets
            yield combined_data, batch_targets

        # After the loop, print and save the missing files
        if missing_files:
            print(f"Missing files: {missing_files}")
            # Append the missing files to a log file
            with open(f"/{Code_run_ID}/{today}_missing_files.log", "a") as log_file:
                for file in missing_files:
                    log_file.write(f"{file}\n")
            missing_files = []  # Clear the list after saving

In [ ]:
# Assuming input shape is (512, 512, 3)
input_shape = (img_size, img_size, 210) #250? - #wdith, height of bounding box, last 204 is number of spectral bands of the Specim IQ camera + 6 cultivar onehot encoding



In [ ]:
# checkpoint_Brix = tf.keras.callbacks.ModelCheckpoint(
#                     filepath=f"{Code_run_ID}_{today}_model_file_brix_subset.keras", 
#                     monitor="val_mae", mode="min", 
#                     save_best_only=True,
#                     save_weights_only=False,
#                     verbose=1)

# checkpoint_firmness = tf.keras.callbacks.ModelCheckpoint(
#                     filepath=f"{Code_run_ID}_{today}_model_file_firmness_subset.keras", 
#                     monitor="val_mae", mode="min", 
#                     save_best_only=True,
#                     save_weights_only=False,
#                     verbose=1)

checkpoint_starch_only = tf.keras.callbacks.ModelCheckpoint(
                    filepath=f"{save_file_path}{Code_run_ID}_{today}_model_file_starch.keras", 
                    monitor="val_mae", mode="min", 
                    save_best_only=True,
                    save_weights_only=False,
                    verbose=1)                    


In [ ]:
strategy = tf.distribute.MirroredStrategy()

with strategy.scope():
    # Create a Sequential model
    model_complex_starch_only = Sequential()

    # Convolutional Block 1
    model_complex_starch_only.add(
        Conv2D(64, kernel_size=(2, 2), activation="relu", input_shape=input_shape) # kernal size format is common choice for image data
    )
    model_complex_starch_only.add(BatchNormalization())
    # model_complex_starch_only.add(MaxPooling2D(pool_size=(2, 2))) # cuts image in half
    model_complex_starch_only.add(Dropout(0.25))

    # Convolutional Block 2
    model_complex_starch_only.add(Conv2D(128, kernel_size=(2, 2), activation="relu"))
    model_complex_starch_only.add(BatchNormalization())
    model_complex_starch_only.add(MaxPooling2D(pool_size=(2, 2), padding = 'same'))
    model_complex_starch_only.add(Dropout(0.25))

    # Convolutional Block 3
    model_complex_starch_only.add(Conv2D(256, kernel_size=(2, 2), activation="relu"))
    model_complex_starch_only.add(BatchNormalization())
    model_complex_starch_only.add(MaxPooling2D(pool_size=(2, 2), padding = 'same'))
    model_complex_starch_only.add(Dropout(0.25))

    # Convolutional Block 4
    model_complex_starch_only.add(Conv2D(512, kernel_size=(2, 2), activation="relu"))
    model_complex_starch_only.add(BatchNormalization())
    model_complex_starch_only.add(MaxPooling2D(pool_size=(2, 2), padding = 'same'))
    model_complex_starch_only.add(Dropout(0.25))

    # Convolutional Block 5
    model_complex_starch_only.add(Conv2D(1024, kernel_size=(2, 2), activation="relu"))
    model_complex_starch_only.add(BatchNormalization())
    model_complex_starch_only.add(MaxPooling2D(pool_size=(2, 2), padding = 'same'))
    model_complex_starch_only.add(Dropout(0.25))

    # Convolutional Block 6
    # model_complex_starch_only.add(Conv2D(2048, kernel_size=(3, 3), activation="relu")) #2048 - number of filters
    # model_complex_starch_only.add(BatchNormalization())
    # model_complex_starch_only.add(MaxPooling2D(pool_size=(2, 2), padding = 'same'))
    # model_complex_starch_only.add(Dropout(0.25))

    # Global Average Pooling
    model_complex_starch_only.add(GlobalAveragePooling2D())

    # Dense layers
    model_complex_starch_only.add(Dense(2048, activation="relu")) # Dense layers other type of extraction method
    model_complex_starch_only.add(BatchNormalization())
    model_complex_starch_only.add(Dropout(0.5))

    model_complex_starch_only.add(Dense(1024, activation="relu"))
    model_complex_starch_only.add(BatchNormalization())
    model_complex_starch_only.add(Dropout(0.5))

    model_complex_starch_only.add(Dense(512, activation="relu"))
    model_complex_starch_only.add(BatchNormalization())
    model_complex_starch_only.add(Dropout(0.5))

    model_complex_starch_only.add(Dense(256, activation="relu"))
    model_complex_starch_only.add(BatchNormalization())
    model_complex_starch_only.add(Dropout(0.5))

    # Output layer with 1 neuron for regression
    model_complex_starch_only.add(Dense(1, activation="linear")) # This dense layer sets number of output nodes
    
    # Compile the model
    model_complex_starch_only.compile(optimizer="adam", loss="mean_squared_error", metrics=["mae"])

model_complex_starch_only.summary()

In [5]:
# load all_data

training_data_path = '/home/ht21074/Auto_box_apples/Auto_box_apples/Source_folder/Training_data_Feb2025/all_years/'

X_train_Starch          = np.load(f'{training_data_path}X_train_all_years_Starch_shuffled.npy')
Y_train_Starch          = np.load(f'{training_data_path}Y_train_all_years_Starch_shuffled.npy')
X_validate_Starch       = np.load(f'{training_data_path}X_validate_all_years_Starch_shuffled.npy')
Y_validate_Starch       = np.load(f'{training_data_path}Y_validate_all_years_Starch_shuffled.npy')
Starch_encoder_shuffled  = np.load(f'{training_data_path}X_train_all_years_Starch_encoder_shuffled.npy')
validate_encoder        = np.load(f'{training_data_path}X_validate_all_years_Starch_encoder_shuffled.npy')


In [6]:
print(Y_train_Starch[:3])

[42. 65. 45.]


In [8]:
spectral_path = '/media/2tbdisk2/data/Haidee_apple_data/Haidee/Data_outputs_NZ2023/Spectral/'

X_train_Starch = [spectral_path + file for file in X_train_Starch]

X_validate_Starch = [spectral_path + file for file in X_validate_Starch]





In [ ]:
# X_train_all_years_Starch_encoder_shuffled = np.load(f'{training_data_path}X_train_all_years_Starch_encoder_shuffled.npy')
# print(X_train_all_years_Starch_encoder_shuffled[0:3])


[[0. 1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0.]]


In [ ]:
# test_X = X_train_Starch[0:4]
# test_Y = Y_train_Starch[0:4]
# test_cultivar = Starch_encoder_shuffled[0:4]

# print(test_X)
# print(test_Y)
# print(test_cultivar)

["/media/2tbdisk2/data/Haidee_apple_data/Haidee/Data_outputs_NZ2023/Spectral/subsetted_aggregated_hyperspectral_images/NZ2023/aggregated_Cox's Orange Pippin_1071_2758.npy", '/media/2tbdisk2/data/Haidee_apple_data/Haidee/Data_outputs_NZ2023/Spectral/subsetted_aggregated_hyperspectral_images/NZ2024/aggregated_Fuji_1325_7897.npy', '/media/2tbdisk2/data/Haidee_apple_data/Haidee/Data_outputs_NZ2023/Spectral/subsetted_aggregated_hyperspectral_images/UK2024/aggregated_Braeburn_1449_18312.npy', "/media/2tbdisk2/data/Haidee_apple_data/Haidee/Data_outputs_NZ2023/Spectral/subsetted_aggregated_hyperspectral_images/NZ2023/aggregated_Cox's Orange Pippin_1085_4466.npy"]
[8. 7. 8. 9.]
[[0. 1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0.]]


In [ ]:
# data_generator_w_cultivar(test_X, test_Y, test_cultivar, 2, img_size=20)



NameError: name 'data_generator_w_cultivar' is not defined

In [14]:
# # Debugging the generator
# file_list = test_X
# batch_size = 2
# img_size=20
# targets = test_Y
# cultivars = test_cultivar

# num_samples = len(file_list)
# missing_files = [] # List of missing files

# while True:  # Infinite loop for generator
#     for offset in range(0, num_samples, batch_size):
#         # Load the batch of data from file paths
#         batch_files = file_list[offset: offset + batch_size]
#         batch_data = []
#         batch_targets = []
#         batch_cultivars = []
#         # File loading and handling - ensures model runs if file not found
#         for i, file in enumerate(batch_files):
#             try:
#                 data = np.load(file)
#                 batch_data.append(data)
#                 batch_targets.append(targets[offset + i])
#                 batch_cultivars.append(cultivars[offset + i])
#             except FileNotFoundError:
#                 missing_files.append(file)
#                 print(f"File not found: {file}. Skipping...")
#                 continue
        
#         # Convert lists to numpy arrays
#         batch_data = np.array(batch_data)  # Shape: (batch_size, 20, 20, 204)
#         batch_targets = np.array(batch_targets)  # Shape: (batch_size,)
#         batch_cultivars = np.array(batch_cultivars)  # Shape: (batch_size, 6)
        
#         if len(batch_data) == 0:
#             continue # Skip if no data loaded (if the all batch files are missing )
        
   
        
#         # Expand cultivar information to match the input data's spatial dimensions
#         expanded_cultivars = np.repeat(batch_cultivars[:, np.newaxis, np.newaxis, :], img_size, axis=1) # Adds singleton dimensions to match the input data's spatial dimensions (batchsize, 1, 1, 1, 6)
#         print(expanded_cultivars.shape)
#         expanded_cultivars = np.repeat(expanded_cultivars, img_size, axis=2)
#         print(expanded_cultivars.shape)
#         print(expanded_cultivars.shape)
#             # Concatenate cultivar information with the original data along the last axis
#         combined_data = np.concatenate([batch_data, expanded_cultivars], axis=-1)  # Shape: (batch_size, 20, 20, 204, 7)
#         print(combined_data.shape)
#         # Make sure combined_data is 4D and has the one hot encoding for the cultivars (batch_size, height, width, channels, cultivar) 
#         # combined_data = np.squeeze(combined_data, axis=-2)  # Shape: (batch_size, 20, 20, 204, 6)
#         # Remove the singleton dimension introduced above to make the final shape 4D
#         # combined_data = np.squeeze(combined_data, axis=-2)  # Shape: (batch_size, 20, 20, 204)
#         # Yield the combined data and targets
#         # yield combined_data, batch_targets
#     # After the loop, print and save the missing files
#     if missing_files:
#         print(f"Missing files: {missing_files}")
#         # Append the missing files to a log file
#         with open(f"{Code_run_ID}_{today}_missing_files.log", "a") as log_file:
#             for file in missing_files:
#                 log_file.write(f"{file}\n")
#         missing_files = []  # Clear the list after saving
#     break  # Exit the generator function

(2, 20, 1, 6)
(2, 20, 20, 6)
(2, 20, 20, 6)
(2, 20, 20, 210)
(2, 20, 1, 6)
(2, 20, 20, 6)
(2, 20, 20, 6)
(2, 20, 20, 210)


In [ ]:
from datetime import datetime
batch_size = 32  # Set your batch size

today = datetime.today().strftime('%Y-%m-%d')
# print(today)

csv_logger = CSVLogger(f"{save_file_path}{Code_run_ID}_{today}history_starch.csv", append=True)

In [ ]:
train_generator = data_generator_w_cultivar(X_train_Starch, Y_train_Starch, Starch_encoder_shuffled, batch_size, img_size=img_size)
val_generator = data_generator_w_cultivar(X_validate_Starch, Y_validate_Starch, validate_encoder, batch_size, img_size=img_size)


In [ ]:
model_complex_starch_only.fit(train_generator,epochs=110,steps_per_epoch=len(X_train_Starch) // batch_size,validation_data=val_generator,validation_steps=len(X_validate_Starch) // batch_size,callbacks=[tf.keras.callbacks.CSVLogger(f"{save_file_path}{Code_run_ID}_{today}history_starch_subset.csv", append=True), checkpoint_starch_only],)


In [ ]:
model_complex_starch_only.save(f'{save_file_path}{Code_run_ID}_{today}model_trained_100epoch_starch.keras')


In [20]:
# del model_complex_starch_only
# gc.collect()
start_time = datetime.datetime.now()
starch_end = datetime.datetime.now()
print(starch_end)
print(starch_end - start_time)
# starch_total_time = starch_end - start_time

2025-02-21 13:39:43.236960
0:00:00.000078
